<h1>Testing SEN3 outputs and results</h1>

<p>Visualising WQSF flags and unmasked science variables<br>
Exports GeoTIFFs for inspection on QGIS<br>
Outputs:<br>
-- WQSF quality flag layers (one band per flag, 0/1 binary)<br>
-- Unmasked science variable layers (CHL_OC4ME, CHL_NN, PAR)<br>
All outputs are reprojected to EPSG:5479 at 300m resolution</p>

In [9]:
import os
import numpy as np
import netCDF4 as nc
import rasterio
from rasterio.transform import from_bounds
from pyresample import geometry, kd_tree
from pyproj import CRS, Transformer

In [10]:
downloads_folder = ("/Users/gwyneth/Desktop/oct2022")
swath_output_folder = ("/Users/gwyneth/Desktop/oct2022_mar2023_masked")
vis_output_folder = ("/Users/gwyneth/Desktop")

target_crs = "EPSG:5479"
target_res_metres = 300

# which WQSF flags to export as individual binary layers
flags_to_export = [
    # water classification
    "WATER",
    "INLAND_WATER",
    # common invalid flags
    "CLOUD",
    "CLOUD_AMBIGUOUS",
    "CLOUD_MARGIN",
    "INVALID",
    "COSMETIC",
    "SATURATED",
    "SUSPECT",
    "HISOLZEN",
    "HIGHGLINT",
    "SNOW_ICE",
    # BAC processing flags
    "AC_FAIL",
    "WHITECAPS",
    "ADJAC",
    "RWNEG_O2", "RWNEG_O3", "RWNEG_O4",
    "RWNEG_O5", "RWNEG_O6", "RWNEG_O7", "RWNEG_O8",
    # variable-specific flags
    "OC4ME_FAIL",
    "OCNN_FAIL",
    "PAR_FAIL",
]

# science variables to export unmasked
science_vars = ["CHL_OC4ME", "CHL_NN"]

# shared resampling helper
# reprojects a 2D array on the native OLCI swath grid to a regular projected grid
# uses the nearest-neighbour approach
def resample_to_geotiff(data_2d, lat, lon, out_path,
                        target_crs=target_crs,
                        resolution_m=target_res_metres,
                        nodata=np.nan,
                        dtype=np.float32):

    transformer = Transformer.from_crs("EPSG:4326", target_crs, always_xy=True)
    x_proj, y_proj = transformer.transform(lon.ravel(), lat.ravel())

    x_min, x_max = x_proj.min(), x_proj.max()
    y_min, y_max = y_proj.min(), y_proj.max()

    grid_cols = int(np.ceil((x_max - x_min) / resolution_m))
    grid_rows = int(np.ceil((y_max - y_min) / resolution_m))

    swath_def = geometry.SwathDefinition(lons=lon, lats=lat)
    crs_obj = CRS.from_epsg(int(target_crs.split(":")[1]))

    area_def = geometry.AreaDefinition(
        area_id = "mslc2000_grid",
        description = f"MSLC2000 {resolution_m}m grid",
        proj_id = target_crs,
        projection = crs_obj.to_dict(),
        width = grid_cols,
        height = grid_rows,
        area_extent = (x_min, y_min, x_max, y_max),
    )

    resampled = kd_tree.resample_nearest(
        swath_def, data_2d.astype(np.float32), area_def,
        radius_of_influence=resolution_m * 2,
        fill_value=np.nan,
    )

    transform = from_bounds(x_min, y_min, x_max, y_max, grid_cols, grid_rows)

    with rasterio.open(
        out_path, "w",
        driver = "GTiff",
        height = grid_rows,
        width = grid_cols,
        count = 1,
        dtype = np.float32,
        crs = target_crs,
        transform = transform,
        nodata = nodata,
        compress = "lzw",
    ) as dst:
        dst.write(resampled.astype(np.float32), 1)

# first export: WQSF flag layers
# reads wqsf.nc from each .SEN3 folder and exports one binary GeoTIFF per flag
# pixel value: 1 = flag is set, 0 = flag not set, NaN = nodata
def export_wqsf_flag_layers(sen3_folder, out_folder):

    wqsf_path = os.path.join(sen3_folder, "wqsf.nc")
    geo_path  = os.path.join(sen3_folder, "geo_coordinates.nc")

    if not os.path.exists(wqsf_path) or not os.path.exists(geo_path):
        print(f"Missing wqsf.nc or geo_coordinates.nc in {sen3_folder}, skipping.")
        return

    granule_name = os.path.basename(sen3_folder)
    granule_out = os.path.join(out_folder, granule_name, "wqsf_flags")
    os.makedirs(granule_out, exist_ok=True)

    with nc.Dataset(geo_path) as geo:
        lat = (geo.variables["latitude"][:].data).astype(np.float32)
        lon = (geo.variables["longitude"][:].data).astype(np.float32)

    with nc.Dataset(wqsf_path) as wqsf_ds:
        wqsf = wqsf_ds.variables["WQSF"][:]
        flag_masks = wqsf_ds.variables["WQSF"].flag_masks
        flag_meanings = wqsf_ds.variables["WQSF"].flag_meanings.split()

        available = set(flag_meanings)
        requested = set(flags_to_export)
        missing = requested - available
        if missing:
            print(f"Note: These flags were not found in wqsf.nc and will be skipped: {missing}")

        for flag_name in flags_to_export:
            if flag_name not in available:
                continue

            out_path = os.path.join(granule_out, f"{flag_name}.tif")
            if os.path.exists(out_path):
                print(f"Exists, skipping: {flag_name}.tif")
                continue

            bit = int(flag_masks[flag_meanings.index(flag_name)])
            flag_layer = ((wqsf & bit) != 0).astype(np.float32) # 1 where set, 0 where not

            resample_to_geotiff(flag_layer, lat, lon, out_path, nodata=np.nan)
            print(f"Exported flag: {flag_name}.tif  (set pixels: {int(flag_layer.sum()):,})")

    print(f"WQSF flag layers written to: {granule_out}")

# second export: unmasked science variable layers
# reads from the unmasked NetCDF with embedded coordinates
def export_unmasked_science_layers(sen3_folder, swath_output_folder, out_folder):

    granule_name = os.path.basename(sen3_folder)
    raw_swath_path = os.path.join(swath_output_folder, f"{granule_name}_qualitymasked.nc")

    if not os.path.exists(raw_swath_path):
        print(f"Raw swath NetCDF not found, run Step 1 first: {raw_swath_path}")
        return

    granule_out = os.path.join(out_folder, granule_name, "masked_vars")
    os.makedirs(granule_out, exist_ok=True)

    with nc.Dataset(raw_swath_path) as ds:
        lat = ds.variables["latitude"][:].astype(np.float32)
        lon = ds.variables["longitude"][:].astype(np.float32)

        for varname in science_vars:
            if varname not in ds.variables:
                print(f"{varname} not found in raw swath, skipping.")
                continue

            out_path = os.path.join(granule_out, f"{varname}_unmasked.tif")
            if os.path.exists(out_path):
                print(f"Exists, skipping: {varname}_qualitymasked.tif")
                continue

            data = ds.variables[varname][:].astype(np.float32)
            if np.ma.is_masked(data):
                data = data.filled(np.nan)

            resample_to_geotiff(data, lat, lon, out_path, nodata=np.nan)

            n_valid = int(np.sum(~np.isnan(data)))
            print(f"Exported unmasked: {varname}_qualitymasked.tif  (non-NaN pixels: {n_valid:,})")

    print(f"Masked science layers written to: {granule_out}")

In [11]:
# main code

# pick any single granule by name
granule_to_check = "S3A_OL_2_WFR____20221030T211002_20221030T211302_20221101T094938_0179_091_285_4320_MAR_O_NT_003.SEN3"

sen3_folder = os.path.join(downloads_folder, granule_to_check)

print("Exporting WQSF flag layers...")
export_wqsf_flag_layers(sen3_folder, vis_output_folder)

print("\nExporting quality-masked science variable layers...")
export_unmasked_science_layers(sen3_folder, swath_output_folder, vis_output_folder)

Exporting WQSF flag layers...


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Exported flag: WATER.tif  (set pixels: 20,224)


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Exported flag: INLAND_WATER.tif  (set pixels: 17)


...

Exported flag: PAR_FAIL.tif  (set pixels: 0)
WQSF flag layers written to: /Users/gwyneth/Desktop/S3A_OL_2_WFR____20221030T211002_20221030T211302_20221101T094938_0179_091_285_4320_MAR_O_NT_003.SEN3/wqsf_flags

Exporting quality-masked science variable layers...


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Exported unmasked: CHL_OC4ME_qualitymasked.tif  (non-NaN pixels: 8,459)


/opt/miniconda3/lib/python3.13/site-packages/pyproj/crs/crs.py:1293: UserWarning: You will likely lose important projection information when converting to a PROJ string from another format. See: https://proj.org/faq.html#what-is-the-best-format-for-describing-coordinate-reference-systems
  proj = self._crs.to_proj4(version=version)


Exported unmasked: CHL_NN_qualitymasked.tif  (non-NaN pixels: 0)
Masked science layers written to: /Users/gwyneth/Desktop/S3A_OL_2_WFR____20221030T211002_20221030T211302_20221101T094938_0179_091_285_4320_MAR_O_NT_003.SEN3/masked_vars
